# TechOps Intelligence Platform
## Notebook 03 — PDF Processing Pipeline

**Author:** Sidhu  
**Phase:** 2 — Document Processing  
**Goal:** Extract, chunk, embed and store PDF content
          into ChromaDB knowledge_base collection

### PDFs Processed
- site_reliability_engineering.pdf (Google SRE Book)
- building_secure_and_reliable_systems.pdf (Google SRE Vol 2)
- sre_workbook.pdf (Google SRE Workbook)
- aws_well_architected.pdf (AWS Architecture Framework)
- aws_genai_lens.pdf (AWS GenAI Best Practices)

### Pipeline Steps
1. Load PDF with PyMuPDF
2. Detect if page is text-based or scanned
3. Extract text (direct or TrOCR fallback)
4. Clean and chunk by section
5. Embed with all-mpnet-base-v2
6. Store in ChromaDB with metadata
7. Test retrieval quality

In [2]:
# SETUP + IMPORTS
# ─────────────────────────────────────────
import os
import re
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

# PDF processing
import fitz  # PyMuPDF

# OCR for scanned pages
from PIL import Image
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel
)
import torch

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector DB
import chromadb
# PROJECT PATHS
# ─────────────────────────────────────────
PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

RAW_PDFS    = PROJECT_ROOT / "data/raw/pdfs"
PROCESSED   = PROJECT_ROOT / "data/processed"
EMBEDDINGS  = PROJECT_ROOT / "data/embeddings"

## 1. Load Models
PyMuPDF for text extraction
TrOCR for scanned page fallback
all-mpnet-base-v2 for embeddings
ChromaDB for storage

In [3]:
# ─────────────────────────────────────────
# LOAD EMBEDDING MODEL
# Reuse same model as Notebook 02
# ─────────────────────────────────────────
print("Loading embedding model...")
embedding_model = SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2',
    device='cpu'
)
print("Embedding model loaded (768-dim)")

# ─────────────────────────────────────────
# LOAD TROCR FOR SCANNED PAGES
# Only loads when scanned page detected
# Lazy loading saves memory
# ─────────────────────────────────────────
trocr_processor = None
trocr_model     = None

def get_trocr():
    """Lazy load TrOCR — only when needed"""
    global trocr_processor, trocr_model
    if trocr_processor is None:
        print("Loading TrOCR for scanned page...")
        trocr_processor = TrOCRProcessor.from_pretrained(
            "microsoft/trocr-base-printed"
        )
        trocr_model = VisionEncoderDecoderModel.from_pretrained(
            "microsoft/trocr-base-printed"
        )
        trocr_model.eval()
        print("TrOCR loaded")
    return trocr_processor, trocr_model


# ─────────────────────────────────────────
# CONNECT TO CHROMADB
# Add to existing persistent store
# ─────────────────────────────────────────
chroma_path = str(EMBEDDINGS / "chroma_db")
client      = chromadb.PersistentClient(path=chroma_path)

# Create knowledge_base collection for PDFs
knowledge_base = client.get_or_create_collection(
    name     = "knowledge_base",
    metadata = {"hnsw:space": "cosine"}
)

print(f"\nChromaDB connected")
print(f"  knowledge_base collection: {knowledge_base.count()} docs")
print(f"  Existing collections: {[c.name for c in client.list_collections()]}")

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2201.29it/s]


Embedding model loaded (768-dim)

ChromaDB connected
  knowledge_base collection: 0 docs
  Existing collections: ['logs', 'playbooks', 'incidents', 'postmortems', 'knowledge_base']


## 2. PDF Processing Functions
Three-stage pipeline per page:
1. Try direct text extraction (fast)
2. If text < threshold → scanned page → TrOCR
3. Clean extracted text

In [4]:
# ─────────────────────────────────────────
# PDF PROCESSING FUNCTIONS
# ─────────────────────────────────────────

def is_scanned_page(page, min_chars: int = 50) -> bool:
    """
    Detect if a PDF page is scanned or text-based.
    Text-based pages have extractable text > min_chars.
    Scanned pages have little or no extractable text.
    """
    text = page.get_text().strip()
    return len(text) < min_chars


def extract_text_direct(page) -> str:
    """
    Extract text directly from text-based PDF page.
    Preserves structure better than raw get_text().
    """
    # Extract with layout preservation
    text = page.get_text("text")
    # Remove excessive whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


def extract_text_ocr(page) -> str:
    """
    Extract text from scanned PDF page using TrOCR.
    Converts page to image first then runs OCR.
    """
    processor, model = get_trocr()

    # Render page as image at 200 DPI
    pix = page.get_pixmap(dpi=200)
    img = Image.frombytes(
        "RGB",
        [pix.width, pix.height],
        pix.samples
    )

    # TrOCR inference
    pixel_values = processor(
        img, return_tensors="pt"
    ).pixel_values

    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return text.strip()


def extract_pdf_text(pdf_path: str) -> list:
    """
    Extract text from all pages of a PDF.
    Returns list of page dicts with text + metadata.
    """
    doc    = fitz.open(pdf_path)
    pages  = []
    stats  = {"direct": 0, "ocr": 0, "empty": 0}

    for page_num in range(len(doc)):
        page = doc[page_num]

        if is_scanned_page(page):
            # Scanned page — use TrOCR
            try:
                text   = extract_text_ocr(page)
                method = "ocr"
                stats["ocr"] += 1
            except Exception as e:
                text   = ""
                method = "failed"
                stats["empty"] += 1
        else:
            # Text-based page — direct extraction
            text   = extract_text_direct(page)
            method = "direct"
            stats["direct"] += 1

        if text.strip():
            pages.append({
                "page_num"   : page_num + 1,
                "text"       : text,
                "method"     : method,
                "char_count" : len(text),
                "pdf_path"   : str(pdf_path)
            })

    doc.close()
    return pages, stats


def clean_pdf_text(text: str) -> str:
    """
    Clean extracted PDF text.
    Removes headers, footers, page numbers.
    Preserves technical content.
    """
    # Remove page numbers (standalone numbers)
    text = re.sub(r'^\d+$', '', text, flags=re.MULTILINE)

    # Remove common PDF artifacts
    text = re.sub(r'\x0c', '\n', text)  # form feed
    text = re.sub(r'\ufeff', '', text)  # BOM

    # Normalise whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)

    # Remove very short lines (likely headers/footers)
    lines       = text.split('\n')
    clean_lines = [
        l for l in lines
        if len(l.strip()) > 3 or l.strip() == ''
    ]
    text = '\n'.join(clean_lines)

    return text.strip()


print("PDF processing functions defined")

# Quick test on first PDF
test_pdf = list(RAW_PDFS.glob("*.pdf"))[0]
print(f"\nTesting on: {test_pdf.name}")
doc = fitz.open(str(test_pdf))
page = doc[5]  # page 6 (skip cover)
is_scanned = is_scanned_page(page)
text_preview = extract_text_direct(page)[:200]
doc.close()

print(f"Is scanned     : {is_scanned}")
print(f"Text preview   :\n{text_preview}")

PDF processing functions defined

Testing on: aws_genai_lens.pdf
Is scanned     : False
Text preview   :
Generative AI Lens
AWS Well-Architected Framework
GENOPS05-BP01 Learn when to customize models ................................................................... 118
Security ........................


## 3. Chunking Strategy
Section-aware chunking for SRE content:
→ Split on chapter/section headings
→ Fallback to RecursiveCharacterTextSplitter
→ Chunk size: 500 chars, overlap: 50
→ Preserve heading context in each chunk

In [ ]:
# ─────────────────────────────────────────
# CHUNKING STRATEGY
# Section-aware for SRE/AWS documents
# ─────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Section heading patterns for SRE content
SECTION_PATTERNS = [
    r'^Chapter \d+',
    r'^\d+\.\d+\s+[A-Z]',    # "1.2 Section Title"
    r'^[A-Z][A-Z\s]{10,}$',  # "ALL CAPS HEADING"
]

def split_into_chunks(
    text     : str,
    chunk_size: int = 500,
    overlap   : int = 50
) -> list:
    """
    Split text into chunks for embedding.
    Uses RecursiveCharacterTextSplitter with
    paragraph-aware splitting.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size         = chunk_size,
        chunk_overlap      = overlap,
        separators         = [
            "\n\n",   # paragraph break first
            "\n",     # line break
            ". ",     # sentence break
            " ",      # word break
            ""        # character (last resort)
        ],
        length_function    = len,
        is_separator_regex = False
    )
    chunks = splitter.split_text(text)
    # Filter very short chunks
    chunks = [c for c in chunks if len(c.strip()) > 50]
    return chunks


def process_pdf_into_chunks(
    pdf_path    : Path,
    chunk_size  : int = 500,
    overlap     : int = 50
) -> list:
    """
    Full pipeline: PDF → pages → clean → chunks
    Returns list of chunk dicts ready for embedding
    """
    print(f"\nProcessing: {pdf_path.name}")
    pages, stats = extract_pdf_text(str(pdf_path))
    print(f"  Pages extracted : {len(pages)}")
    print(f"  Method breakdown: {stats}")

    all_chunks = []
    for page in pages:
        clean_text = clean_pdf_text(page['text'])
        if len(clean_text) < 50:
            continue

        chunks = split_into_chunks(
            clean_text,
            chunk_size,
            overlap
        )

        for j, chunk in enumerate(chunks):
            all_chunks.append({
                "text"    : chunk,
                "page_num": page['page_num'],
                "method"  : page['method'],
                "source"  : pdf_path.name,
                "chunk_id": j
            })

    print(f"  Total chunks    : {len(all_chunks)}")
    total_chars = sum(len(c['text']) for c in all_chunks)
    print(f"  Total chars     : {total_chars:,}")
    print(f"  Avg chunk size  : {total_chars//max(len(all_chunks),1)} chars")

    return all_chunks


# Test chunking on one PDF
test_chunks = process_pdf_into_chunks(
    RAW_PDFS / "aws_genai_lens.pdf"
)
print(f"\nSample chunk:")
print(f"  {test_chunks[10]['text'][:300]}")

ModuleNotFoundError: No module named 'langchain.text_splitter'